# 29 - Star clipping, star by star: does the colour constraint predict which stars clip?

**Purpose.** Test the half of the model that has never met a real star: the star-colour
constraint. A star's brightest pixel lands at

```
peak = pedestal + (F_star_peak + F_sky + D) * t / g(gain)        ADC counts
```

(`model.peak_counts`), and the star's colour is lost when that reaches the ceiling. MISSION's
deliverable is the Pareto curve between SNR and star colour; the SNR half passed its test in
`27`. This notebook asks whether the other half predicts the right stars.

**The test.** One list of stars for all four cells of the sky-pair night (`data/session06`).
Each star's peak is measured in cell A (gain 50, 30 s), which clips least. From that the model
gives the star's `F_star_peak`, and from that it predicts the star's peak in B, C and D. Then the
peaks and the clipping are measured in B, C and D and set against the prediction. It is rule 7 of
`protocols/06-sky-pair.md`: the fraction of stars with a clipped core, per plane, per cell, against
the same detection list.

**What it pins down.** Whether star peaks scale with `t` and with `1/g` the way `peak_counts`
says - that is the whole of the constraint's arithmetic, and `g` at gains 50 and 200 is `15`'s
cold value. And whether the response stays straight up to where the model's ceiling sits:
`linear_to_at_least` was proved on a flat field, never on a star core.

**How the stars are followed, and why not through PixInsight.** Registration resamples, and
resampling smooths a peak - the one number measured here. So nothing here is registered. Each raw
sub is lined up with the reference by **whole-pixel shifts**, one per tile of a 4 x 4 grid
(`spatial.tile_offsets`, `spatial.realign`). A whole-pixel shift moves no value. One shift per
frame is not enough: the field turns over the night and the corners drift about 2 px against the
centre. Per tile, every star lands within about a pixel of its listed core, and its peak is the
brightest pixel within 1 px of it (`stats.peaks`). The reference is `25`'s rule: the cell D frame
nearest the middle of the post-flip night.

**The list.** Stars found on a deep image - the mean of every cell D sub, lined up the same way,
greens averaged - by `spatial.stars`: 5 sigma above a 31-px running median, **isolated** (no other
star within 5 px), so a neighbour's core never sits in the box. It is not every star in this
crowded field, and the fractions below are fractions of this list.

**A star's peak in a cell** is the median over that cell's subs. One sub's peak scatters with
seeing and with where the star falls on the pixel; the median over 36 subs does not, and the
interleave spreads seeing evenly over the four cells. A star is **clipped in a cell** if its
median peak is 4095, i.e. it clips in at least half its subs.

**Fixed before any peak is seen.**

1. **Peak scaling, per cell and plane, B, C and D.** Stars that stand at least **50 counts** above
   the sky in A, and whose *predicted* peak is below 0.9 x `linear_to_at_least` - selected on A and
   on the prediction only, never on what is measured in the cell, which would cut the high side
   off. The ratio is measured over predicted, both above the pedestal. **It passes if the median
   ratio is within 10% of 1**, MISSION's tolerance.
2. **Clipped fraction, per cell and plane, B, C and D.** The measured fraction of the list that
   clips, against a band: the fraction the model predicts with every predicted peak (above the
   pedestal) scaled by 0.9 and by 1.1. **It passes if the measured fraction lies inside the band.**
   A star clipped in A is predicted clipped everywhere: every other cell puts more counts on it.

The constraint **holds on this night** if all twelve cell-planes pass both. If not, the way it
fails says where:

- **B fails and C does not** - B is gain 50 at 120 s, so `g` cancels and only `t` moved. A star's
  peak is not growing as `t`: a longer sub blurs it more, through guiding and seeing. Then
  `F_star_peak` depends on `t`, and the model's constraint is too pessimistic for long subs.
- **C fails and B does not** - only the gain moved, so `g`'s ratio between 50 and 200 is wrong at a
  star peak, though `15` measured it on flats.
- **The ratio falls as the predicted peak nears the ceiling** - the response bends before 4095 on
  a star core, and `linear_to_at_least` is not conservative enough.

**What it publishes.** `results/star_peaks.csv`, one row per star, plane and cell, and
`results/star_constants.json`: the peak scale, the clipped fractions measured and predicted, the
verdict, and `F_star_peak` as measured in A - the quantity `model.t_max_colour` takes. That last
one belongs to **this target, this night's seeing and this focus**, and is labelled so.

**What it is not for.** Not the Pareto curve itself: that is the next notebook's job, once this
one says whether the constraint can be trusted. Not colour accuracy, star shapes or any
processing. Not the SNR: the masked pairs in `27` stand. Not a new star mask for `27`.

**It assumes** `17_sky_pair.ipynb` for the night - which frames are good, `F_sky` per cell, and the
white-balance gate, which every frame of this session passed - and `15_cold_constants.ipynb` for
`g` and the pedestal at -20 C.

In [ ]:
import json
import pathlib
import sys
import time

import numpy as np
import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import fits, model as M, spatial, stats

ROOT = pathlib.Path.cwd().parent
RESULTS = ROOT / "results"
DATA = ROOT / "data" / "session06"

sky = json.loads((RESULTS / "sky_constants.json").read_text())
cold = json.loads((RESULTS / "cold_constants.json").read_text())
mc = json.loads((RESULTS / "model_constants.json").read_text())
F_SKY = sky["F_sky_per_cell"]["value"]                  # e-/px/s, per cell and plane
G = {int(k): v for k, v in cold["system_gain_cold_per_plane"]["value"].items()}
PEDESTAL = {int(k): v for k, v in cold["pedestal_cold"]["value"].items()}
CEILING = {int(k): v for k, v in mc["linear_to_at_least"]["value"].items()}
DARK = mc["dark_current_bound"]["value"]                # -10 C bound; 27 showed it is nothing

CELLS = {"A": (50, 30.0), "B": (50, 120.0), "C": (200, 30.0), "D": (200, 120.0)}
REF_CELL = "A"
PLANES = spatial.PLANES
FULL_SCALE = stats.ADC_FULL_SCALE
TILES = (4, 4)
RADIUS = 1
DETECT = {"nsigma": 5.0, "background": 31, "isolation": 5}
MIN_EXCESS = 50.0              # counts above the sky in A, for the scaling test
HEADROOM = 0.9                 # predicted peak below this x linear_to_at_least, same test
TOLERANCE = 0.10               # MISSION's

PEAKS_CSV = RESULTS / "star_peaks.csv"
CONSTANTS = RESULTS / "star_constants.json"
print("g:", G, "\npedestal:", PEDESTAL, "\nlinear_to_at_least:", CEILING)

## 1. The frames, the reference, and how far the field moves

`17`'s good frames, `25`'s reference. Every frame's tile offsets against it, on G1; the other
three planes sit on the same grid, so they share them. The spread between a frame's tiles is the
field turning.

*The first run of this cell found a tile off by 50-230 px in eight cell A subs - faint frames,
nearly always the same starless patch - and a margin of 227 px that cut a third off every plane.
That was a failed correlation, not a movement, and it was found from the spread printed here, not
from any peak. `spatial.tile_offsets` now fits each frame's shift and turn over its tiles and
replaces a tile that disagrees, and the notebook was re-run from the top under the same rules.*

In [ ]:
frames = pd.read_csv(RESULTS / "sky_frames.csv", parse_dates=["t"])
good = frames[~frames.flipped & ~frames.rejected].sort_values("t").reset_index(drop=True)
mid = good.t.min() + (good.t.max() - good.t.min()) / 2
d = good[good.cell == "D"]
REF = d.loc[(d.t - mid).abs().idxmin()]
print(f"{len(good)} frames; reference {REF.file}")
print(good.groupby("cell").size().rename("subs").to_frame().T.to_string())


def planes_of(file):
    mosaic, _ = fits.read(DATA / file)
    return spatial.split(stats.to_adc(mosaic))


ref_g1 = planes_of(REF.file)["G1"]
t0 = time.time()
offsets = {f: spatial.tile_offsets(ref_g1, planes_of(f)["G1"], TILES) for f in good.file}
spread = pd.Series({f: int(np.ptp(o.reshape(-1, 2), axis=0).max()) for f, o in offsets.items()})
MARGIN = int(max(np.abs(o).max() for o in offsets.values()))
print(f"offsets in {time.time() - t0:.0f} s; largest {MARGIN} px, so the margin")
print("spread between one frame's tiles, px:", spread.value_counts().sort_index().to_dict())

## 2. The star list

The deep image is the mean of every cell D sub, realigned, G1 and G2 averaged. Stars within the
peak box of its edge are dropped.

In [ ]:
deep = None
for f in good.file[good.cell == "D"]:
    p = planes_of(f)
    g = (spatial.realign(p["G1"], offsets[f], MARGIN).astype(np.float64)
         + spatial.realign(p["G2"], offsets[f], MARGIN)) / 2
    deep = g if deep is None else deep + g
deep /= (good.cell == "D").sum()

STARS = spatial.stars(deep, **DETECT)
h, w = deep.shape
inside = ((STARS >= RADIUS) & (STARS < np.array([h, w]) - RADIUS)).all(axis=1)
STARS = STARS[inside]
print(f"deep image {deep.shape}; {len(STARS)} isolated stars on the list")

## 3. Every star's peak in every sub

One pass over the night: each frame is realigned plane by plane and every star's peak read. About
a second a frame.

In [ ]:
t0 = time.time()
rows = []
for r in good.itertuples():
    p = planes_of(r.file)
    for plane in PLANES:
        pk = stats.peaks(spatial.realign(p[plane], offsets[r.file], MARGIN), STARS, RADIUS)
        rows.append(pd.DataFrame({"star": np.arange(len(STARS)), "cell": r.cell,
                                  "plane": plane, "file": r.file, "peak": pk}))
subs = pd.concat(rows, ignore_index=True)
print(f"{len(subs)} star-subs in {time.time() - t0:.0f} s")

per = (subs.assign(clipped=subs.peak >= FULL_SCALE)
       .groupby(["star", "plane", "cell"])
       .agg(n_subs=("peak", "size"), peak_median=("peak", "median"),
            clipped_sub_frac=("clipped", "mean"))
       .reset_index())
per["clipped"] = per.peak_median >= FULL_SCALE

## 4. What the model predicts

From each star's median peak in A, `F_star_peak` is what is left after the pedestal, the sky and
the dark are taken off, in electrons per second. `model.peak_counts` then puts it into B, C and
D. A star already clipped in A has only a lower bound, and is predicted clipped everywhere.

In [ ]:
gain_a, t_a = CELLS[REF_CELL]
a = (per[per.cell == REF_CELL][["star", "plane", "peak_median", "clipped"]]
     .rename(columns={"peak_median": "peak_in_A", "clipped": "clipped_in_A"}))
a["F_star_peak"] = ((a.peak_in_A - PEDESTAL[gain_a]) * a.plane.map(G[gain_a]) / t_a
                    - a.plane.map(F_SKY[REF_CELL]) - DARK)
a["excess_in_A"] = (a.peak_in_A - PEDESTAL[gain_a]
                    - a.plane.map(F_SKY[REF_CELL]) * t_a / a.plane.map(G[gain_a]))
per = per.merge(a, on=["star", "plane"])

per["peak_predicted"] = np.nan
for cell, (gain, t) in CELLS.items():
    at = per.cell == cell
    per.loc[at, "peak_predicted"] = M.peak_counts(
        per.F_star_peak[at], t, f_sky=per.plane[at].map(F_SKY[cell]), dark=DARK,
        pedestal=PEDESTAL[gain], g=per.plane[at].map(G[gain]))
per["pedestal"] = per.cell.map(lambda c: PEDESTAL[CELLS[c][0]])
per["ceiling"] = per.cell.map(lambda c: CEILING[CELLS[c][0]])
per["ratio"] = (per.peak_median - per.pedestal) / (per.peak_predicted - per.pedestal)
others = per[per.cell != REF_CELL]
print("stars clipped in A, per plane:", a.groupby("plane").clipped_in_A.sum().to_dict())

## 5. Test 1: do star peaks scale as the model says?

In [ ]:
usable = others[~others.clipped_in_A & (others.excess_in_A >= MIN_EXCESS)
                & (others.peak_predicted < HEADROOM * others.ceiling)]


def mad_se(x):                 # standard error of a median, from the MAD
    return 1.2533 * stats.MAD_TO_SIGMA * np.median(np.abs(x - np.median(x))) / np.sqrt(len(x))


scale = (usable.groupby(["cell", "plane"]).ratio
         .agg(n="size", median="median", se=mad_se,
              q25=lambda x: x.quantile(0.25), q75=lambda x: x.quantile(0.75))
         .reset_index())
scale["pass"] = (scale["median"] - 1).abs() <= TOLERANCE
print(scale.round(4).to_string(index=False))

### Near the ceiling

The verdict above uses only stars predicted well under the ceiling. Here every star the model
predicts unclipped is binned by where its predicted peak lands, as a fraction of 4095. A response
that bends before the top shows as a ratio that falls in the last bins. This is a diagnostic, not
part of the verdict.

In [ ]:
near = others[~others.clipped_in_A & (others.excess_in_A >= MIN_EXCESS)
              & (others.peak_predicted < FULL_SCALE)].copy()
near["level"] = pd.cut(near.peak_predicted / FULL_SCALE, [0, 0.25, 0.5, 0.75, 0.9, 0.95, 1.0])
print(near.groupby(["cell", "level"], observed=True).ratio
      .agg(n="size", median="median").unstack("cell").round(3).to_string())

## 6. Test 2: does the model predict how many stars clip?

In [ ]:
def frac_predicted(df, k):
    lifted = df.pedestal + k * (df.peak_predicted - df.pedestal)
    return float((df.clipped_in_A | (lifted >= FULL_SCALE)).mean())


clip = []
for (cell, plane), df in others.groupby(["cell", "plane"]):
    clip.append({"cell": cell, "plane": plane, "stars": len(df),
                 "measured": float(df.clipped.mean()),
                 "predicted": frac_predicted(df, 1.0),
                 "low": frac_predicted(df, 1 - TOLERANCE),
                 "high": frac_predicted(df, 1 + TOLERANCE)})
clip = pd.DataFrame(clip)
clip["pass"] = (clip.measured >= clip.low) & (clip.measured <= clip.high)
print((clip.assign(**{c: 100 * clip[c] for c in ("measured", "predicted", "low", "high")})
       .round(3).to_string(index=False)))
print("\n(fractions in %)")

## 7. The verdict

In [ ]:
verdict = scale[["cell", "plane", "pass"]].merge(clip[["cell", "plane", "pass"]],
                                                 on=["cell", "plane"], suffixes=("_scale", "_clip"))
verdict["verdict"] = np.where(verdict.pass_scale & verdict.pass_clip, "pass", "fail")
print(verdict.to_string(index=False))
HOLDS = bool((verdict.verdict == "pass").all() and len(verdict) == 12)
print("\nthe clipping constraint", "HOLDS on this night" if HOLDS else "does NOT hold on this night")

## 8. `F_star_peak` in this field

What `model.t_max_colour` takes, as measured in A, per plane, for the stars not clipped there.
The brightest stars clip even in A, so the top of this distribution is cut off, and the count of
those is given beside it.

In [ ]:
QUANTILES = (0.5, 0.9, 0.99, 0.999)
fsp = {p: {**{f"q{q:g}": round(float(g[~g.clipped_in_A].F_star_peak.quantile(q)), 3)
              for q in QUANTILES},
           "n": len(g), "clipped_in_A": int(g.clipped_in_A.sum())}
       for p, g in a.groupby("plane")}
print(pd.DataFrame(fsp).to_string())

## 9. Publish

In [ ]:
measured_on = str(good.t.min().date())


def constant(value, unit, uncertainty, note):
    return {"value": value, "unit": unit, "uncertainty": uncertainty,
            "source_frames": int(len(good)), "measured_on": measured_on,
            "notebook": "29_star_clipping.ipynb", "note": note}


def nest(df, col):
    return {c: {p: round(float(v), 6) for p, v in g.set_index("plane")[col].items()}
            for c, g in df.groupby("cell")}


star_xy = pd.DataFrame(STARS + MARGIN, columns=["row", "col"]).rename_axis("star").reset_index()
out = per.merge(star_xy, on="star")[["star", "row", "col", "plane", "cell", "n_subs",
                                     "peak_median", "clipped_sub_frac", "clipped",
                                     "peak_predicted", "ratio"]]
out.round(4).to_csv(PEAKS_CSV, index=False)

settings = (f"one list of {len(STARS)} isolated stars (spatial.stars {DETECT}) on the mean of the "
            f"cell D subs; peaks by stats.peaks, radius {RADIUS}, on raw subs realigned by "
            f"whole-pixel shifts per {TILES[0]}x{TILES[1]} tile; a star's peak in a cell is the "
            f"median over its subs, and it is clipped there if that median is {FULL_SCALE}.  "
            f"Predictions from each star's peak in cell {REF_CELL}, through model.peak_counts with "
            f"15's cold g and pedestal and 17's F_sky per cell")
constants = {
    "star_peak_scale": constant(
        nest(scale, "median"), "measured / predicted star peak above the pedestal, median over stars",
        nest(scale, "se"),
        f"[tests model.peak_counts: t and 1/g scaling at star peaks] stars at least {MIN_EXCESS} "
        f"counts above the sky in A and predicted below {HEADROOM} x linear_to_at_least.  "
        f"Passes within {TOLERANCE:.0%}.  {settings}"),
    "clipped_star_fraction": constant(
        nest(clip, "measured"), "fraction of the star list whose median peak is full scale", None,
        f"rule 7 of protocols/06-sky-pair.md.  Cell A is the reference, so it is not here.  "
        f"{settings}"),
    "clipped_star_fraction_predicted": constant(
        nest(clip, "predicted"), "fraction of the star list the model predicts clipped",
        {"low": nest(clip, "low"), "high": nest(clip, "high")},
        f"the band is every predicted peak scaled by 1 -/+ {TOLERANCE}.  {settings}"),
    "star_clipping_verdict": constant(
        {f"{r.cell} {r.plane}": r.verdict for r in verdict.itertuples()},
        "pass or fail per cell and plane", None,
        "pass if the peak scale is within the tolerance AND the measured clipped fraction is "
        "inside the predicted band, both fixed in the purpose cell before any peak was read"),
    "F_star_peak": constant(
        fsp, "e-/px/s at a star's brightest plane pixel, quantiles over the star list", None,
        f"[feeds model.t_max_colour] measured in cell {REF_CELL} for stars not clipped there; the "
        f"brightest are clipped even in A, so the top of the distribution is cut off and "
        f"clipped_in_A counts them.  THIS TARGET, THIS NIGHT'S SEEING AND THIS FOCUS: a different "
        f"field or a softer night moves every number.  {settings}"),
}
with open(CONSTANTS, "w") as f:
    json.dump(constants, f, indent=2)
M.Constants.load(CONSTANTS)       # the provenance gate
print("wrote", PEAKS_CSV.name, f"({len(out)} rows),", CONSTANTS.name)